# 03. Признаки, baseline и валидация

Ноутбук собирает признаки без утечки будущих продаж и сравнивает несколько простых baseline-моделей по `net_sales_qty`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.baselines import (
    median_by_weekday,
    moving_average_7,
    moving_average_28,
    naive_last_value,
    seasonal_naive_7,
    seasonal_naive_28,
)
from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.features import build_feature_matrix
from src.metrics import metrics_table

In [ ]:
mart_daily_sales = pd.read_csv(PROCESSED_DATA_DIR / 'mart_daily_sales.csv', parse_dates=['sales_date'])
features = build_feature_matrix(mart_daily_sales)
features.to_csv(PROCESSED_DATA_DIR / 'features_lags_rolling.csv', index=False)
features.head()

## Baseline-модели

Все прогнозы используют только прошлые значения внутри пары `stock_code × market_id`.

In [ ]:
baseline_frame = features[['sales_date', 'stock_code', 'market_id', 'net_sales_qty']].copy()
baseline_frame['naive_last_value'] = naive_last_value(baseline_frame)
baseline_frame['seasonal_naive_7'] = seasonal_naive_7(baseline_frame)
baseline_frame['seasonal_naive_28'] = seasonal_naive_28(baseline_frame)
baseline_frame['moving_average_7'] = moving_average_7(baseline_frame)
baseline_frame['moving_average_28'] = moving_average_28(baseline_frame)
baseline_frame['median_by_weekday'] = median_by_weekday(baseline_frame)
baseline_frame.head(10)

## Расчет метрик

In [ ]:
metric_rows = []
baseline_columns = [
    'naive_last_value',
    'seasonal_naive_7',
    'seasonal_naive_28',
    'moving_average_7',
    'moving_average_28',
    'median_by_weekday',
]

for baseline_name in baseline_columns:
    valid = baseline_frame.dropna(subset=['net_sales_qty', baseline_name])
    table = metrics_table(valid['net_sales_qty'], valid[baseline_name])
    table['baseline'] = baseline_name
    metric_rows.append(table)

baseline_metrics = pd.concat(metric_rows, ignore_index=True)
baseline_metrics = baseline_metrics[['baseline', 'metric', 'value']]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
baseline_metrics.to_csv(RESULTS_DIR / 'baseline_metrics.csv', index=False)
baseline_metrics

## Выводы

- Лучший baseline по WMAPE: `[A]`.
- Если forecast bias положительный, прогноз систематически завышает спрос; если отрицательный, занижает.
- Baseline нужен как честная точка сравнения: более сложный подход имеет смысл только если он устойчиво лучше простых правил.